Treinar e explorar o comportamento de diferentes componentes que integram uma Rede Neural Convolucional (CNN), usando o dataset CIFAR-10 (https://www.cs.toronto.edu/~kriz/cifar.html). Aproveite
os notebooks compartilhados em aula para construir sua solução.

In [13]:
# Imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import Sequential
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.losses import CategoricalCrossentropy

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import ssl

In [2]:
# SSL para resolver problema em baixar CIFAR-10
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

In [3]:
# Importando CIFAR-10
(x_train_val, y_train_val), (x_test, y_test) = cifar10.load_data()


**1. (0,5) Separe 20% dos dados de treinamento para validação. Todos os treinamentos intermediários devem ser avaliados neste conjunto de validação.**

In [31]:
x_train, x_val, y_train, y_val = train_test_split(x_train_val, y_train_val,
                                                    test_size=.2,
                                                    random_state=42)

print("Treino    X: ", x_train.shape)
print("Treino    Y: ", y_train.shape)
print("Validação X: ", x_val.shape)
print("Validação Y: ", y_val.shape)

Treino    X:  (40000, 32, 32, 3)
Treino    Y:  (40000, 1)
Validação X:  (10000, 32, 32, 3)
Validação Y:  (10000, 1)


In [32]:
# preprocessing
x_train = x_train / 255.0
x_val   = x_val / 255.0
x_test  = x_test / 255.0 

In [38]:
# One Hot Encoding
num_classes = len(np.unique(y_train))
y_ohe_train = keras.utils.to_categorical(y_train, num_classes)
y_ohe_val = keras.utils.to_categorical(y_val, num_classes)
y_ohe_test = keras.utils.to_categorical(y_test, num_classes)
print(x_train.shape, y_ohe_train.shape)

(40000, 32, 32, 3) (40000, 10)


In [34]:
# Balanceamento
# Conjunto de Treino
unique, counts = np.unique(y_train, return_counts=True)
print("Treino ---> ", dict(zip(unique, counts)), "\n")

class_weights = compute_class_weight('balanced', np.unique(y_train), y_train[:,0])
train_class_weights = dict(enumerate(class_weights))
print(train_class_weights, "\n")

# Conjunto de Teste
unique, counts = np.unique(y_test, return_counts=True)
print("Teste  ---> ", dict(zip(unique, counts)), "\n")

Treino --->  {0: 4027, 1: 4021, 2: 3970, 3: 3977, 4: 4067, 5: 3985, 6: 4004, 7: 4006, 8: 3983, 9: 3960} 

{0: 0.9932952570151478, 1: 0.9947774185525988, 2: 1.0075566750629723, 3: 1.0057832537088258, 4: 0.9835259404966806, 5: 1.0037641154328734, 6: 0.999000999000999, 7: 0.9985022466300549, 8: 1.0042681395932713, 9: 1.0101010101010102} 

Teste  --->  {0: 1000, 1: 1000, 2: 1000, 3: 1000, 4: 1000, 5: 1000, 6: 1000, 7: 1000, 8: 1000, 9: 1000} 



**2. (1,0) Construa uma rede neural convolucional para baseline. Sugestão: comece por uma arquitetura simples. Quanto mais simples, melhor.**

In [35]:
model = Sequential()
#Camada convolucional com 10 filtros de tamanho 3x3 e ativação ReLU
model.add(layers.Conv2D(10, 3, padding='valid', activation='relu', input_shape=(32, 32, 3)))
#Max pooling de tamanho 2x2
model.add(layers.MaxPooling2D(pool_size=(2,2)))
#Operação de vetorização dos dados
model.add(layers.Flatten())
#Dropout com probabilidade de 0.2
model.add(layers.Dropout(0.2))
#Densa com 10 nós de saída
model.add(layers.Dense(10))

model.summary()

Model: "sequential_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d_1 (Conv2D)            (None, 30, 30, 10)        280       
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 15, 15, 10)        0         
_________________________________________________________________
flatten_1 (Flatten)          (None, 2250)              0         
_________________________________________________________________
dropout_1 (Dropout)          (None, 2250)              0         
_________________________________________________________________
dense_1 (Dense)              (None, 10)                22510     
Total params: 22,790
Trainable params: 22,790
Non-trainable params: 0
_________________________________________________________________


In [36]:
model.compile(optimizer=tf.keras.optimizers.Adam(0.01),
              loss=CategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [41]:
model.fit(
    x=x_train, 
    y=y_ohe_train, 
    epochs=30, 
    batch_size=32,
    class_weight=train_class_weights,
    validation_data=(x_val, y_ohe_val))

Epoch 1/30
1250/1250 [==============================] - 7s 6ms/step - loss: 1.7188 - accuracy: 0.3884 - val_loss: 1.6674 - val_accuracy: 0.4020
Epoch 2/30
1250/1250 [==============================] - 7s 6ms/step - loss: 1.7113 - accuracy: 0.3918 - val_loss: 1.8070 - val_accuracy: 0.3572
Epoch 3/30
1250/1250 [==============================] - 8s 6ms/step - loss: 1.7198 - accuracy: 0.3852 - val_loss: 1.6878 - val_accuracy: 0.4026
Epoch 4/30
1250/1250 [==============================] - 9s 7ms/step - loss: 1.7097 - accuracy: 0.3889 - val_loss: 1.6669 - val_accuracy: 0.4128
Epoch 5/30
1250/1250 [==============================] - 9s 7ms/step - loss: 1.7155 - accuracy: 0.3885 - val_loss: 1.6303 - val_accuracy: 0.4195
Epoch 6/30
1250/1250 [==============================] - 9s 7ms/step - loss: 1.7200 - accuracy: 0.3899 - val_loss: 1.7189 - val_accuracy: 0.3761
Epoch 7/30
1250/1250 [==============================] - 9s 7ms/step - loss: 1.7127 - accuracy: 0.3868 - val_loss: 1.7006 - val_accuracy:

**3. (0,5) Explore o impacto de três diferentes funções de ativação.**

In [ ]:
#sigmoid

In [42]:
model_sigmoid = Sequential()
#Camada convolucional com 10 filtros de tamanho 3x3 e ativação ReLU
model_sigmoid.add(layers.Conv2D(10, 3, padding='valid', activation='sigmoid', input_shape=(32, 32, 3)))
#Max pooling de tamanho 2x2
model_sigmoid.add(layers.MaxPooling2D(pool_size=(2,2)))
#Operação de vetorização dos dados
model_sigmoid.add(layers.Flatten())
#Dropout com probabilidade de 0.2
model_sigmoid.add(layers.Dropout(0.2))
#Densa com 10 nós de saída
model_sigmoid.add(layers.Dense(10))

model_sigmoid.summary()

Model: "sequential_2"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d_2 (Conv2D)            (None, 30, 30, 10)        280       
_________________________________________________________________
max_pooling2d_2 (MaxPooling2 (None, 15, 15, 10)        0         
_________________________________________________________________
flatten_2 (Flatten)          (None, 2250)              0         
_________________________________________________________________
dropout_2 (Dropout)          (None, 2250)              0         
_________________________________________________________________
dense_2 (Dense)              (None, 10)                22510     
Total params: 22,790
Trainable params: 22,790
Non-trainable params: 0
_________________________________________________________________


In [43]:
model_sigmoid.compile(optimizer=tf.keras.optimizers.Adam(0.01),
              loss=CategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [44]:
model_sigmoid.fit(
    x=x_train, 
    y=y_ohe_train, 
    epochs=30, 
    batch_size=32,
    class_weight=train_class_weights,
    validation_data=(x_val, y_ohe_val))

Epoch 1/30
1250/1250 [==============================] - 8s 6ms/step - loss: 2.1067 - accuracy: 0.2973 - val_loss: 1.7991 - val_accuracy: 0.3726
Epoch 2/30
1250/1250 [==============================] - 8s 6ms/step - loss: 1.7504 - accuracy: 0.3858 - val_loss: 1.6819 - val_accuracy: 0.4196
Epoch 3/30
1250/1250 [==============================] - 8s 7ms/step - loss: 1.6586 - accuracy: 0.4197 - val_loss: 1.6111 - val_accuracy: 0.4252
Epoch 4/30
1250/1250 [==============================] - 9s 8ms/step - loss: 1.5793 - accuracy: 0.4493 - val_loss: 1.5659 - val_accuracy: 0.4696
Epoch 5/30
1250/1250 [==============================] - 10s 8ms/step - loss: 1.5166 - accuracy: 0.4674 - val_loss: 1.4727 - val_accuracy: 0.4981
Epoch 6/30
1250/1250 [==============================] - 9s 8ms/step - loss: 1.4668 - accuracy: 0.4890 - val_loss: 1.4747 - val_accuracy: 0.4846
Epoch 7/30
1250/1250 [==============================] - 9s 8ms/step - loss: 1.4151 - accuracy: 0.5089 - val_loss: 1.3952 - val_accuracy

In [ ]:
#TahnLu

In [45]:
model_tanh = Sequential()
#Camada convolucional com 10 filtros de tamanho 3x3 e ativação ReLU
model_tanh.add(layers.Conv2D(10, 3, padding='valid', activation='tanh', input_shape=(32, 32, 3)))
#Max pooling de tamanho 2x2
model_tanh.add(layers.MaxPooling2D(pool_size=(2,2)))
#Operação de vetorização dos dados
model_tanh.add(layers.Flatten())
#Dropout com probabilidade de 0.2
model_tanh.add(layers.Dropout(0.2))
#Densa com 10 nós de saída
model_tanh.add(layers.Dense(10))

model_tanh.summary()

Model: "sequential_3"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d_3 (Conv2D)            (None, 30, 30, 10)        280       
_________________________________________________________________
max_pooling2d_3 (MaxPooling2 (None, 15, 15, 10)        0         
_________________________________________________________________
flatten_3 (Flatten)          (None, 2250)              0         
_________________________________________________________________
dropout_3 (Dropout)          (None, 2250)              0         
_________________________________________________________________
dense_3 (Dense)              (None, 10)                22510     
Total params: 22,790
Trainable params: 22,790
Non-trainable params: 0
_________________________________________________________________


In [46]:
model_tanh.compile(optimizer=tf.keras.optimizers.Adam(0.01),
              loss=CategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [47]:
model_tanh.fit(
    x=x_train, 
    y=y_ohe_train, 
    epochs=30, 
    batch_size=32,
    class_weight=train_class_weights,
    validation_data=(x_val, y_ohe_val))

Epoch 1/30
1250/1250 [==============================] - 10s 8ms/step - loss: 1.8016 - accuracy: 0.3891 - val_loss: 1.5426 - val_accuracy: 0.4630
Epoch 2/30
1250/1250 [==============================] - 9s 7ms/step - loss: 1.6573 - accuracy: 0.4464 - val_loss: 1.6149 - val_accuracy: 0.4521
Epoch 3/30
1250/1250 [==============================] - 10s 8ms/step - loss: 1.6148 - accuracy: 0.4651 - val_loss: 1.9185 - val_accuracy: 0.3966
Epoch 4/30
1250/1250 [==============================] - 10s 8ms/step - loss: 1.6153 - accuracy: 0.4727 - val_loss: 1.5650 - val_accuracy: 0.4947
Epoch 5/30
1250/1250 [==============================] - 10s 8ms/step - loss: 1.6121 - accuracy: 0.4764 - val_loss: 1.5571 - val_accuracy: 0.4778
Epoch 6/30
1250/1250 [==============================] - 9s 7ms/step - loss: 1.5848 - accuracy: 0.4832 - val_loss: 1.4554 - val_accuracy: 0.5077
Epoch 7/30
1250/1250 [==============================] - 9s 8ms/step - loss: 1.5777 - accuracy: 0.4873 - val_loss: 1.9216 - val_accur

4. (2,5) Explore o impacto de variar a quantidade de camadas de convolução e pooling. Cuidado com
overfitting! Explore, no mínimo, duas arquiteturas diferentes do baseline.

5. (1,5) Explore o uso de duas diferentes inicializações e regularizações.

6. (0,5) Avalie o uso de dropout na camada totalmente conectada.

7. (1,0) Plote os gráficos da função de loss × número de épocas, para o treino e validação de cada
modelo. Houve overfitting?

8. (0,5) A partir dos experimentos acima, construa o que você considera o melhor modelo e faça a
avaliação no conjunto de teste. Obs: Avaliação no conjunto de teste só pode ser executada uma
única vez.

9. (2,0) Sumarize os seus resultados e conclusões em relação aos resultados experimentais.